In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
from matplotlib import rcParams
import time
from datetime import datetime
import itertools
from collections import defaultdict

# PyTorch and PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv, global_mean_pool, BatchNorm
from torch_geometric.data import Data, DataLoader
from torch_geometric.utils import dense_to_sparse

# Sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings
import warnings
import gc
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Graph Transformer Grid Search started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# HELPER FUNCTIONS

In [ ]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    """Convert time series data to supervised learning format."""
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    
    # Input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    
    # Forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [ ]:
def create_spatial_graph(data, k_neighbors=3, correlation_threshold=0.2):
    """Create spatial graph structure based on feature relationships."""
    # Calculate correlation matrix
    corr_matrix = np.corrcoef(data.T)
    
    # Create adjacency matrix with top-k neighbors and correlation threshold
    n_features = corr_matrix.shape[0]
    adj_matrix = np.zeros_like(corr_matrix)
    
    for i in range(n_features):
        # Get correlations for node i
        correlations = np.abs(corr_matrix[i])
        
        # Find top-k neighbors above threshold
        top_indices = np.argsort(correlations)[::-1][:k_neighbors+1]  # +1 for self
        for j in top_indices:
            if correlations[j] > correlation_threshold:
                adj_matrix[i, j] = correlations[j]
    
    # Ensure symmetry
    adj_matrix = (adj_matrix + adj_matrix.T) / 2
    np.fill_diagonal(adj_matrix, 1.0)  # Self-connections
    
    # Convert to edge list
    edge_index, edge_weights = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float))
    
    return edge_index, edge_weights, corr_matrix

In [ ]:
def create_positional_encoding(max_len, d_model):
    """Create positional encoding for temporal sequences."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                       -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    if d_model % 2 == 1:
        pe[:, 1::2] = torch.cos(position * div_term[:-1])
    else:
        pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

In [ ]:
def create_graph_sequence_data(X, Y, seq_length=12):
    """Convert time series data to graph sequence format."""
    graph_data_list = []
    
    # Create base graph structure from feature correlations
    sample_data = X[0]  # [timesteps, features]
    edge_index, edge_attr, _ = create_spatial_graph(sample_data)
    
    for i in range(len(X)):
        # Create sequence of temporal features
        sequence_data = []
        
        for t in range(seq_length):
            if t < X[i].shape[0]:
                # Node features at time t - convert numpy to tensor
                node_features = torch.tensor(X[i][t], dtype=torch.float32).reshape(-1, 1)  # [features, 1]
            else:
                # Padding for shorter sequences
                node_features = torch.zeros(X[i].shape[1], 1)
            
            sequence_data.append(node_features)
        
        # Stack temporal features: [features, seq_length]
        node_features = torch.cat(sequence_data, dim=1)
        
        # Create graph data object
        data = Data(
            x=node_features.float(),
            edge_index=edge_index.long(),
            edge_attr=edge_attr.float(),
            y=torch.tensor([Y[i]], dtype=torch.float32),
            seq_length=torch.tensor([min(seq_length, X[i].shape[0])], dtype=torch.long)
        )
        graph_data_list.append(data)
    
    return graph_data_list

# LOAD DATASET

In [ ]:
print("Loading dataset...")
dataset = pd.read_csv("eMalahleniIM.csv", sep=';', header=0, index_col=0)
values = dataset.values
print(f"Dataset shape: {dataset.shape}")

# DATA PREPROCESSING

In [ ]:
print("Preprocessing data...")

# Ensure all data is float
values = values.astype('float32')

# Normalize features
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)
n_features = scaled.shape[1]

# GRAPH TRANSFORMER MODEL COMPONENTS

In [ ]:
class MultiHeadGraphAttention(nn.Module):
    """Multi-head graph attention layer for spatial modeling."""
    
    def __init__(self, in_dim, out_dim, num_heads=8, dropout=0.1):
        super(MultiHeadGraphAttention, self).__init__()
        self.num_heads = num_heads
        self.out_dim = out_dim
        self.head_dim = out_dim // num_heads
        
        assert self.head_dim * num_heads == out_dim
        
        self.transformers = nn.ModuleList([
            TransformerConv(in_dim, self.head_dim, heads=1, dropout=dropout)
            for _ in range(num_heads)
        ])
        
        self.output_projection = nn.Linear(out_dim, out_dim)
        self.layer_norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_attr=None):
        # Multi-head attention (ignore edge_attr to avoid dimension mismatch)
        head_outputs = []
        for transformer in self.transformers:
            head_out = transformer(x, edge_index)  # Remove edge_attr parameter
            head_outputs.append(head_out)
        
        # Concatenate heads
        multi_head_out = torch.cat(head_outputs, dim=-1)
        
        # Output projection and residual connection
        out = self.output_projection(multi_head_out)
        out = self.layer_norm(out + x if x.size(-1) == self.out_dim else out)
        out = self.dropout(out)
        
        return out

In [ ]:
class TemporalTransformerLayer(nn.Module):
    """Temporal transformer layer for time series modeling."""
    
    def __init__(self, d_model, num_heads=8, d_ff=256, dropout=0.1):
        super(TemporalTransformerLayer, self).__init__()
        
        self.self_attention = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention
        attn_out, _ = self.self_attention(x, x, x, attn_mask=mask)
        x = self.layer_norm1(x + self.dropout(attn_out))
        
        # Feed-forward
        ff_out = self.feed_forward(x)
        x = self.layer_norm2(x + self.dropout(ff_out))
        
        return x

In [ ]:
class GraphTransformerModel(nn.Module):
    """Graph Transformer Network for air pollution prediction."""
    
    def __init__(self, n_features, seq_length, d_model=128, 
                 num_graph_layers=2, num_temporal_layers=2, 
                 num_heads=8, dropout=0.1):
        super(GraphTransformerModel, self).__init__()
        
        self.n_features = n_features
        self.seq_length = seq_length
        self.d_model = d_model
        
        # Input projection
        self.input_projection = nn.Linear(seq_length, d_model)
        
        # Positional encoding
        self.register_buffer('pos_encoding', create_positional_encoding(n_features, d_model))
        
        # Spatial graph attention layers
        self.graph_layers = nn.ModuleList([
            MultiHeadGraphAttention(d_model, d_model, num_heads, dropout)
            for _ in range(num_graph_layers)
        ])
        
        # Temporal transformer layers
        self.temporal_layers = nn.ModuleList([
            TemporalTransformerLayer(d_model, num_heads, d_model*2, dropout)
            for _ in range(num_temporal_layers)
        ])
        
        # Feature aggregation - will be dynamically adjusted
        self.feature_aggregation = nn.Sequential(
            nn.Linear(d_model * n_features, d_model),  # This will be adjusted dynamically
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Final prediction head
        self.prediction_head = nn.Sequential(
            nn.Linear(d_model // 2, d_model // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1),
            nn.Sigmoid()
        )
    
    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        batch_size = batch.max().item() + 1
        
        # Input projection: [nodes, seq_length] -> [nodes, d_model]
        x = self.input_projection(x)
        
        # Add positional encoding - ensure size compatibility
        pos_enc_size = min(x.size(0), self.pos_encoding.size(0))
        x[:pos_enc_size] = x[:pos_enc_size] + self.pos_encoding[:pos_enc_size]
        
        # Spatial modeling with graph attention
        for graph_layer in self.graph_layers:
            x = graph_layer(x, edge_index, edge_attr)
        
        # Reshape for temporal modeling: [batch_size, num_nodes, d_model]
        nodes_per_graph = x.size(0) // batch_size
        x_temporal = x.view(batch_size, nodes_per_graph, self.d_model)
        
        # Temporal modeling with transformer
        for temporal_layer in self.temporal_layers:
            x_temporal = temporal_layer(x_temporal)
        
        # Feature aggregation: flatten and aggregate node features
        x_flat = x_temporal.view(batch_size, -1)
        
        # Handle dynamic input size for feature aggregation
        if hasattr(self, '_dynamic_aggregation') and x_flat.size(1) != self.feature_aggregation[0].in_features:
            # Recreate aggregation layer with correct input size
            self.feature_aggregation = nn.Sequential(
                nn.Linear(x_flat.size(1), self.d_model),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.d_model, self.d_model // 2),
                nn.ReLU(),
                nn.Dropout(0.1)
            ).to(x.device)
        elif x_flat.size(1) != self.feature_aggregation[0].in_features:
            # First time - adjust the layer
            self.feature_aggregation = nn.Sequential(
                nn.Linear(x_flat.size(1), self.d_model),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(self.d_model, self.d_model // 2),
                nn.ReLU(),
                nn.Dropout(0.1)
            ).to(x.device)
            self._dynamic_aggregation = True
        
        x_aggregated = self.feature_aggregation(x_flat)
        
        # Final prediction
        out = self.prediction_head(x_aggregated)
        
        return out.squeeze()

# TRAINING FUNCTIONS

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    predictions = []
    targets = []
    
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            out = model(batch)
            loss = criterion(out, batch.y)
            
            total_loss += loss.item()
            predictions.extend(out.cpu().numpy())
            targets.extend(batch.y.cpu().numpy())
    
    return total_loss / len(data_loader), np.array(predictions), np.array(targets)


In [ ]:
def train_and_evaluate_model(params, train_data, val_data, n_features, device, max_epochs=50):
    """Train and evaluate a single model configuration."""
    
    try:
        # Create data loaders
        train_loader = DataLoader(train_data, batch_size=params['batch_size'], shuffle=True)
        val_loader = DataLoader(val_data, batch_size=params['batch_size'], shuffle=False)
        
        # Initialize model
        model = GraphTransformerModel(
            n_features=n_features,
            seq_length=params['seq_length'],
            d_model=params['d_model'],
            num_graph_layers=params['num_graph_layers'],
            num_temporal_layers=params['num_temporal_layers'],
            num_heads=params['num_heads'],
            dropout=params['dropout']
        ).to(device)
        
        # Loss function and optimizer
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
        
        # Training loop
        best_val_loss = float('inf')
        patience = 8
        patience_counter = 0
        
        for epoch in range(max_epochs):
            # Train
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            
            # Validate
            val_loss, _, _ = evaluate_model(model, val_loader, criterion, device)
            
            # Learning rate scheduling
            scheduler.step()
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                break
        
        # Clean up memory
        del model, optimizer, scheduler, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return best_val_loss
        
    except Exception as e:
        print(f"    Training failed: {e}")
        # Clean up on error
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float('inf')  # Return high loss for failed runs

# GRID SEARCH PARAMETERS

In [ ]:
print("Setting up grid search parameters...")

# Define parameter grid (smaller grid due to model complexity)
param_grid = {
    'seq_length': [6, 12, 24],
    'd_model': [64, 128, 256],
    'num_graph_layers': [1, 2, 3],
    'num_temporal_layers': [1, 2, 3],
    'num_heads': [4, 8, 16],
    'dropout': [0.1, 0.2, 0.3],
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [8, 16, 32]
}

print("Parameter combinations to test:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nTotal combinations: {total_combinations}")

# GRID SEARCH EXECUTION

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Storage for results
results = []
best_score = float('inf')
best_params = None

print(f"\nStarting grid search...")
start_time = time.time()

In [ ]:
combination_count = 0

for seq_length in param_grid['seq_length']:
    print(f"\n--- Testing seq_length: {seq_length} ---")
    
    # Create sequence data for current sequence length
    X_sequences = []
    Y_sequences = []
    
    for i in range(len(scaled) - seq_length):
        X_sequences.append(scaled[i:i+seq_length])
        Y_sequences.append(scaled[i+seq_length, 0])  # Predict PM2.5
    
    X = np.array(X_sequences)
    Y = np.array(Y_sequences)
    
    # Convert to graph sequence data
    graph_data = create_graph_sequence_data(X, Y, seq_length)
    
    # Split data
    n_samples = len(graph_data)
    indices = list(range(n_samples))
    train_val_idx, _ = train_test_split(indices, test_size=0.2, random_state=42)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=42)
    
    train_data = [graph_data[i] for i in train_idx]
    val_data = [graph_data[i] for i in val_idx]
    
    # Test other parameters with this sequence length
    other_params = {k: v for k, v in param_grid.items() if k != 'seq_length'}
    param_combinations = list(itertools.product(*other_params.values()))
    
    for combination in param_combinations:
        combination_count += 1
        
        # Create parameter dictionary
        params = dict(zip(other_params.keys(), combination))
        params['seq_length'] = seq_length
        
        print(f"Testing combination {combination_count}/{total_combinations}: {params}")
        
        try:
            # Train and evaluate
            val_loss = train_and_evaluate_model(params, train_data, val_data, n_features, device)
            
            # Store results
            result = params.copy()
            result['val_loss'] = val_loss
            result['r2_score'] = 1 - val_loss  # Approximate R2 from MSE loss
            results.append(result)
            
            # Update best parameters
            if val_loss < best_score:
                best_score = val_loss
                best_params = params.copy()
                print(f"  New best! Val Loss: {val_loss:.6f}")
            else:
                print(f"  Val Loss: {val_loss:.6f}")
                
        except Exception as e:
            print(f"  Error: {e}")
            continue

total_time = time.time() - start_time
print(f"\nGrid search completed in {total_time:.2f} seconds")

# RESULTS ANALYSIS

In [ ]:
print("\n" + "="*60)
print("GRAPH TRANSFORMER GRID SEARCH RESULTS")
print("="*60)

if not results:
    print("\n❌ No successful training runs were completed!")
    print("All parameter combinations failed during training.")
    print("This might be due to:")
    print("  - Tensor dimension mismatches")
    print("  - Memory issues")
    print("  - Model architecture incompatibilities")
    print("  - Data preprocessing problems")
else:
    print(f"\n✅ Successfully completed {len(results)} training runs")
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('val_loss')

    print(f"\nBest Parameters:")
    if best_params:
        print(f"Val Loss: {best_score:.6f}")
        for param, value in best_params.items():
            print(f"  {param}: {value}")
    else:
        print("❌ No best parameters found!")

    print(f"\nTop 10 Results:")
    print(results_df.head(10)[['seq_length', 'd_model', 'num_graph_layers', 'num_temporal_layers', 
                              'num_heads', 'dropout', 'learning_rate', 'batch_size', 'val_loss']].to_string(index=False))


# VISUALIZATIONS

In [ ]:
if results:  # Only create visualizations if we have results
    print("\nCreating visualizations...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()

    # 1. Sequence Length vs Performance
    seq_results = results_df.groupby('seq_length')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[0].errorbar(seq_results['seq_length'], seq_results['mean'], 
                    yerr=seq_results['std'], marker='o', linewidth=2)
    axes[0].set_xlabel('Sequence Length', fontweight='bold')
    axes[0].set_ylabel('Validation Loss', fontweight='bold')
    axes[0].set_title('Sequence Length vs Performance', fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    # 2. Model Dimension vs Performance
    d_model_results = results_df.groupby('d_model')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[1].errorbar(d_model_results['d_model'], d_model_results['mean'], 
                    yerr=d_model_results['std'], marker='o', linewidth=2)
    axes[1].set_xlabel('Model Dimension', fontweight='bold')
    axes[1].set_ylabel('Validation Loss', fontweight='bold')
    axes[1].set_title('Model Dimension vs Performance', fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    # 3. Graph Layers vs Performance
    graph_layers_results = results_df.groupby('num_graph_layers')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[2].errorbar(graph_layers_results['num_graph_layers'], graph_layers_results['mean'], 
                    yerr=graph_layers_results['std'], marker='o', linewidth=2)
    axes[2].set_xlabel('Number of Graph Layers', fontweight='bold')
    axes[2].set_ylabel('Validation Loss', fontweight='bold')
    axes[2].set_title('Graph Layers vs Performance', fontweight='bold')
    axes[2].grid(True, alpha=0.3)

    # 4. Temporal Layers vs Performance
    temporal_layers_results = results_df.groupby('num_temporal_layers')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[3].errorbar(temporal_layers_results['num_temporal_layers'], temporal_layers_results['mean'], 
                    yerr=temporal_layers_results['std'], marker='o', linewidth=2)
    axes[3].set_xlabel('Number of Temporal Layers', fontweight='bold')
    axes[3].set_ylabel('Validation Loss', fontweight='bold')
    axes[3].set_title('Temporal Layers vs Performance', fontweight='bold')
    axes[3].grid(True, alpha=0.3)

    # 5. Number of Heads vs Performance
    heads_results = results_df.groupby('num_heads')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[4].errorbar(heads_results['num_heads'], heads_results['mean'], 
                    yerr=heads_results['std'], marker='o', linewidth=2)
    axes[4].set_xlabel('Number of Attention Heads', fontweight='bold')
    axes[4].set_ylabel('Validation Loss', fontweight='bold')
    axes[4].set_title('Attention Heads vs Performance', fontweight='bold')
    axes[4].grid(True, alpha=0.3)

    # 6. Dropout vs Performance
    dropout_results = results_df.groupby('dropout')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[5].errorbar(dropout_results['dropout'], dropout_results['mean'], 
                    yerr=dropout_results['std'], marker='o', linewidth=2)
    axes[5].set_xlabel('Dropout Rate', fontweight='bold')
    axes[5].set_ylabel('Validation Loss', fontweight='bold')
    axes[5].set_title('Dropout Rate vs Performance', fontweight='bold')
    axes[5].grid(True, alpha=0.3)

    # 7. Learning Rate vs Performance
    lr_results = results_df.groupby('learning_rate')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[6].errorbar(lr_results['learning_rate'], lr_results['mean'], 
                    yerr=lr_results['std'], marker='o', linewidth=2)
    axes[6].set_xlabel('Learning Rate', fontweight='bold')
    axes[6].set_ylabel('Validation Loss', fontweight='bold')
    axes[6].set_title('Learning Rate vs Performance', fontweight='bold')
    axes[6].set_xscale('log')
    axes[6].grid(True, alpha=0.3)

    # 8. Batch Size vs Performance
    batch_results = results_df.groupby('batch_size')['val_loss'].agg(['mean', 'std']).reset_index()
    axes[7].errorbar(batch_results['batch_size'], batch_results['mean'], 
                    yerr=batch_results['std'], marker='o', linewidth=2)
    axes[7].set_xlabel('Batch Size', fontweight='bold')
    axes[7].set_ylabel('Validation Loss', fontweight='bold')
    axes[7].set_title('Batch Size vs Performance', fontweight='bold')
    axes[7].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('graph_transformer_grid_search_results.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("❌ Skipping visualizations due to no successful results.")

# SAVE RESULTS

In [ ]:
if results:
    # Save detailed results
    results_df.to_csv('graph_transformer_grid_search_results.csv', index=False)

    # Save best parameters
    if best_params:
        best_params_df = pd.DataFrame([best_params])
        best_params_df['best_val_loss'] = best_score
        best_params_df.to_csv('graph_transformer_best_parameters.csv', index=False)

    print(f"\n✅ Graph Transformer Grid Search completed successfully!")
    print(f"📁 Results saved:")
    print(f"  📊 Detailed results: 'graph_transformer_grid_search_results.csv'")
    if best_params:
        print(f"  🏆 Best parameters: 'graph_transformer_best_parameters.csv'")
    print(f"  📈 Visualizations: 'graph_transformer_grid_search_results.png'")
else:
    print(f"\n❌ Grid Search completed but no successful results to save.")
    print(f"Consider debugging the model architecture or data preprocessing.")

print(f"  ⏱️ Total time: {total_time:.2f} seconds")